[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter20_your_first_deep_learning_project/notebook.ipynb)

# Chapter 20: Your First Deep Learning Project

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Every chapter so far taught one idea in isolation, on data chosen to make that idea visible. Real projects do not arrive that way. They arrive as a vague goal and a pile of data, and the work is deciding, in order, what success means, what to build, what to tune, what to trust and what to tell people. This chapter does a whole project from start to finish and uses almost everything in the book along the way.

The data is the handwritten digits from Chapter 17, all ten of them, and this time the job is not a demonstration but a model you could hand to someone else. That choice is deliberate. You already know the data, so every page of this chapter can go on the process rather than on explaining a new dataset, and the process is what transfers to your own projects.

Expect the code here to take a few minutes in total, mostly in the search over network sizes. Before you read on, try to guess the answers to these four questions.

1. **Before training any network, what numbers do you need in order to know whether it worked?**
2. **How would you choose a network's width with evidence instead of habit?**
3. **When two settings differ by one test example in three hundred, which one is better?**
4. **What does someone else need from you to trust and reuse your model?**

## Choosing a Problem and Data

Start by writing down what success means, before any network exists. Here the task is to read a handwritten digit from an 8 by 8 scan, the measure is accuracy on digits the model has never seen, and success means beating the best simple method by a margin worth the extra complexity. That last clause is the one people skip, and it is the one that keeps a project honest.

> 🖼️ **Figure 20.1:** The shape of the project. All the looping happens between design and validation; the test set is touched a single time, after every decision is made.

Split the data three ways, as Chapter 15 did and Book 5 insisted. The test fifth is set aside now and not looked at again until the last section.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
digits = load_digits()
X, y = digits.images / 16.0, digits.target
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=0.25, random_state=7, stratify=y_rest)
print(len(X_train), len(X_val), len(X_test))

Now the baselines, the numbers a network has to beat to be worth building. The first is the know nothing answer, always predicting the most common digit. The second is the plain logistic regression of Book 4, run on the 64 pixel values as if they were ordinary columns.

In [ ]:
flat = lambda a: a.reshape(len(a), -1)
majority = np.bincount(y_train).argmax()
print("always guess", majority, round(np.mean(y_val == majority), 3))
logreg = LogisticRegression(max_iter=2000).fit(flat(X_train), y_train)
print("logistic regression", round(logreg.score(flat(X_val), y_val), 3))

Ten classes in equal numbers make the know nothing score 0.1, as expected. The logistic regression scores 0.956 on validation, which is a strong result from a model with no hidden layers at all. That is the real bar. A network that scores 0.96 has not earned its place; Book 5's lesson that a simpler model often wins applies here too. What would earn it is cutting the remaining errors substantially, and with 360 validation digits, 0.956 means about sixteen mistakes.

## Designing the Network

Chapter 17 showed that convolution suits images, and Chapter 16 gave the recipe for a network that trains reliably. The builder below combines them, with the two numbers you might want to change, the width and the number of blocks, as arguments. Each block doubles the number of channels and halves the size of the map.

In [ ]:
T = lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(1)
Xtr, Xva, Xte = T(X_train), T(X_val), T(X_test)
ytr, yva, yte = torch.tensor(y_train), torch.tensor(y_val), torch.tensor(y_test)
loss_fn = nn.CrossEntropyLoss()

def build(width=8, blocks=2, seed=0):
    torch.manual_seed(seed)
    layers, channels, size = [], 1, 8
    for b in range(blocks):
        out = width * 2 ** b
        layers += [nn.Conv2d(channels, out, 3, padding=1),
                   nn.BatchNorm2d(out), nn.ReLU(), nn.MaxPool2d(2)]
        channels, size = out, size // 2
    layers += [nn.Flatten(), nn.Linear(channels * size * size, 10)]
    return nn.Sequential(*layers)

The training function is the full version from Part 4: Adam, shuffled batches with a seeded generator, and early stopping on validation loss with the best weights restored, as in Chapter 15.

In [ ]:
def fit(net, epochs=40, batch=32, lr=0.01, seed=0, patience=None):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    best, best_state, best_epoch, waited = 1e9, None, 0, 0
    for epoch in range(epochs):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(net(xb), yb).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            val = loss_fn(net(Xva), yva).item()
        if val < best:
            best, best_epoch, waited = val, epoch, 0
            best_state = copy.deepcopy(net.state_dict())
        else:
            waited += 1
            if patience and waited >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        accuracy = (net(Xva).argmax(1) == yva).float().mean().item()
    return round(accuracy, 3), best_epoch

Before tuning anything, check that the extra machinery is worth it at all, by giving a plain dense network from Chapter 17 the same training.

In [ ]:
torch.manual_seed(0)
dense = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(),
                      nn.Linear(64, 10))
print("dense", fit(dense)[0], sum(p.numel() for p in dense.parameters()))

The dense network barely moves past logistic regression: 0.967 against 0.956, about four fewer mistakes, for 4,810 weights. If that were the best available, the honest recommendation would be the logistic regression. The convolutional network has to do clearly better than this to justify itself.

## Choosing Width, Depth and Batch Size

Chapter 11 sized its network by eye and promised that this chapter would choose a width deliberately. Here is how. Train the convolutional network at five widths, three seeds each, and judge every width by its average validation accuracy rather than a single run.

In [ ]:
for width in [2, 4, 8, 16, 32]:
    scores = [fit(build(width, seed=s), seed=s)[0] for s in range(3)]
    count = sum(p.numel() for p in build(width).parameters())
    print(width, count, scores, round(np.mean(scores), 3))

> 🖼️ **Figure 20.2:** Validation accuracy against width. Accuracy climbs steeply up to 16 channels and then flattens; doubling to 32 triples the weights for nothing.

Read Figure 20.2 from left to right. Two channels is too narrow and falls below logistic regression. Four is already better than every baseline. Eight and sixteen keep improving, and thirty two adds three and a half times the weights with no gain at all. The curve has a knee, and the knee is the choice: width 16, the narrowest network at the top of the curve. That is what choosing a width deliberately means. You do not need the largest network that works; you need the smallest one past which more size stops paying.

The other choices get the same treatment, at width 16, still on validation only.

In [ ]:
def average(blocks=2, **settings):
    return round(np.mean([fit(build(16, blocks, seed=s), seed=s, **settings)[0]
                          for s in range(3)]), 3)

print("one block   ", average(blocks=1))
print("two blocks  ", average(blocks=2))
print("batch 16    ", average(batch=16))
print("batch 128   ", average(batch=128))
print("rate 0.003  ", average(lr=0.003))
print("rate 0.03   ", average(lr=0.03))

This is the most useful result in the chapter, and it is the dullest table in it. Every setting lands between 0.985 and 0.992, and the default of two blocks, batches of 32 and a rate of 0.01 scored 0.988. On 360 validation digits those gaps are one or two digits. Look back at the width table for scale: three seeds of the very same width 16 network ranged from 0.986 to 0.992, which is as wide as every difference here. These settings are indistinguishable from noise.

The disciplined response is to stop. Picking batch 16 because it printed 0.992 would be choosing a lucky number, and the validation set would quietly become part of the training. Keep the simple defaults, which Part 4 already gave reasons for, and record that you checked.

**Table 20.1: The design decisions and what each rests on. Every entry was decided on validation data; the test set has not yet been used.**

| Decision | Choice | Evidence |
|---|---|---|
| Model family | Convolutional network | Dense 0.967 against logistic regression 0.956 |
| Width | 16 channels, then 32 | The knee of the width curve |
| Depth | Two blocks | One and two blocks tied |
| Batch size | 32 | 16 to 128 within seed noise |
| Learning rate | 0.01 with Adam | 0.003 to 0.03 within seed noise |
| Stopping | Best validation loss, patience 10 | Chapter 15 |

## Training and Tuning Honestly

With every decision made, train the final model. Train it five times from five seeds rather than once, because Chapter 19 showed that a single seed's number is one draw from a spread. Each run may go to 60 epochs but stops once validation loss has failed to improve for 10.

The rules for this section are short, and they are the ones most often broken. Nothing changes after this point. If the test result is disappointing, you report it; you do not go back and adjust the network to fix it, because then the test set has become a second validation set and its number means nothing.

In [ ]:
finals = []
for seed in range(5):
    net = build(16, seed=seed)
    val_acc, stopped_at = fit(net, epochs=60, seed=seed, patience=10)
    finals.append(net)
    print(seed, val_acc, stopped_at)

The second column is each run's best epoch, and it varies from single digits to the forties. That is early stopping doing its job: each run trains as long as it is still improving, rather than for a number of epochs fixed in advance.

## Evaluating and Explaining

Now, once, the test set.

In [ ]:
tests = []
for net in finals:
    with torch.no_grad():
        tests.append(round((net(Xte).argmax(1) == yte).float().mean().item(), 3))
print(tests)
print(round(np.mean(tests), 3), min(tests), max(tests))
print("logistic regression", round(logreg.score(flat(X_test), y_test), 3))

Averaged over five seeds, the network scores 0.991 on the test digits, with individual runs between 0.986 and 0.997. Logistic regression, measured on the same test set, scores 0.964. In mistakes per 360 digits, that is about three against thirteen, and that is the claim worth reporting: roughly four times fewer errors than the best simple method, consistently across seeds. The test numbers are also close to the validation numbers, which is reassuring evidence that the tuning did not overfit the validation set.

Accuracy alone does not say which mistakes remain. A **confusion matrix** counts, for every true digit, what the model said.

In [ ]:
from sklearn.metrics import confusion_matrix

model = finals[0]
with torch.no_grad():
    probs = torch.softmax(model(Xte), dim=1)
predicted = probs.argmax(1)
print(confusion_matrix(yte, predicted))

Rows are the true digit and columns the prediction, so everything off the diagonal is a mistake. There are three, and every one of them involves a 5: a 5 read as a 0, a 5 read as a 4, and a 9 read as a 5. Look at the actual images before guessing why.

In [ ]:
wrong = torch.nonzero(predicted != yte).flatten().tolist()
for i in wrong:
    print(i, "true", yte[i].item(), "said", predicted[i].item(),
          "confidence", round(probs[i].max().item(), 3),
          "for the truth", round(probs[i, yte[i]].item(), 3))

> 🖼️ **Figure 20.3:** All three test mistakes of the first final model, drawn from their real pixels. The middle one is a hesitant error of the kind Chapter 11 described; the first is a readable 9 called a 5 with almost total confidence.

Two of these are understandable. The middle 5 has an unusually closed lower loop, and the model said 0 with only 0.78 confidence, so it knew it was unsure. The 5 on the right has a long crossbar that looks like the arm of a 4. The first one is different. It is labelled 9, and to the eye it is a perfectly readable 9, with a heavy, filled in head and a tail running down. The model called it a 5 with 0.998 confidence and gave the right answer 0.001. Nothing in the image explains that certainty. With only about a hundred training examples of each digit, a 9 written with its loop inked solid may simply be unlike any 9 the network was shown.

That is worth saying plainly in any report, because it marks the limit of what the probabilities can tell you. Chapter 11's model hesitated on its mistakes, which is what a well behaved model does on genuinely ambiguous inputs, and it would be tempting to flag low confidence predictions for a person to check. This example shows why that is not enough on its own: here one mistake in three gave no warning at all. The next step is to collect more examples of the kind the model gets wrong, not to make the network bigger.

## Presenting Your Work

A result nobody else can check or reuse is not finished. Save the final model with everything Chapter 19 said a checkpoint needs, including the design and the preprocessing, since this file is useless to someone who does not know that pixels were divided by 16 and that the network has width 16 and two blocks.

In [ ]:
torch.save({"model": finals[0].state_dict(),
            "design": {"width": 16, "blocks": 2},
            "preprocess": "pixel values divided by 16, shape (1, 8, 8)",
            "classes": list(range(10)),
            "test_accuracy_mean_of_5_seeds": round(float(np.mean(tests)), 3)},
           "digits_cnn.pt")
reloaded = build(**torch.load("digits_cnn.pt")["design"])
reloaded.load_state_dict(torch.load("digits_cnn.pt")["model"])
reloaded.eval()
with torch.no_grad():
    print(torch.equal(reloaded(Xte).argmax(1), predicted))

The reloaded model gives exactly the same 360 predictions, built from nothing but the file and the `build` function. Then write the result down in a form someone can act on. A good summary fits in a paragraph and makes no claim the evidence does not support, like this one.

> *A two block convolutional network with 6,186 weights reads 8 by 8 handwritten digits with 99.1 percent test accuracy, averaged over five training seeds (range 98.6 to 99.7), against 96.4 percent for logistic regression on the same split, roughly four times fewer errors. Width was chosen as the knee of a validation curve; depth, batch size and learning rate made no measurable difference. The remaining errors all involve the digit 5, and one of them, a readable 9 called a 5, was made with 99.8 percent confidence, so confidence scores alone cannot be relied on to flag mistakes, and further gains are more likely to come from more varied training examples than from a larger network.*

Every number in that paragraph came out of a cell in this chapter, each claim names its evidence, and the limitations are stated rather than left for a reader to discover. Table 20.2 turns the chapter into a checklist for your own projects.

**Table 20.2: A checklist for any deep learning project, distilled from this chapter.**

| Step | The question to answer |
|---|---|
| Define success | What measure, on what unseen data, beating what? |
| Baselines | What does know nothing score, and the best simple model? |
| Design | Which recipe from Chapter 16 fits this kind of data? |
| Tune | Which choices change validation results by more than seed noise? |
| Test once | What does the locked test set say, averaged over seeds? |
| Explain | Which mistakes remain, and are they the model's or the data's? |
| Present | Could someone reload, rerun and check every number you quote? |

## Practice and Review

This chapter's practice is a project of your own, so it has fewer exercises and they are larger.

### Try It Yourself

1. **Run the checklist on new data.** Take scikit-learn's `load_breast_cancer`, a table rather than an image, and work through Table 20.2 with a dense network. Be ready for the honest answer that a simpler model may win.
2. **Look at more of the data.** Plot every training 9 and look for ones with a solid, filled in head like the mistaken test digit. How many are there?
3. **Spend the test set properly.** Pretend the test result had been 0.97. Write the paragraph you would have to report, without changing the model.
4. **Present it.** Write a one page report of this project for someone who has not read this book, using only numbers the chapter produced.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why compute baselines before building any network?**
2. **How was the width of the final network chosen?**
3. **Why did the chapter keep the default batch size even though another one printed a higher score?**
4. **Why train the final model from five seeds?**
5. **Why is a confidently wrong prediction more worrying than a hesitant one?**
6. **What must a saved model carry besides its weights?**

### Answers

1. Because a result only means something next to what was already possible. Logistic regression's 0.956 set the bar, and the dense network's 0.967 barely cleared it.
2. By training five widths on three seeds each and taking the narrowest width at the top of the validation curve, which was 16.
3. Because every difference was one or two digits out of 360, no larger than the spread between seeds of one setting, so choosing the highest would have been choosing noise.
4. Because one seed gives one draw from a spread; five give an average and a range that describe the method rather than one lucky run.
5. Because it gives no warning. A hesitant mistake can be caught by flagging low confidence predictions; a confident one cannot, so it points to a gap in the training data that has to be found by looking at examples.
6. The design needed to rebuild the network, the preprocessing applied to inputs, the meaning of the outputs, and the result it achieved.

## What Comes Next

You have now carried a deep learning project from a definition of success to a saved, reloaded, explained model and a paragraph you could defend in front of anyone. That is the whole book in one chapter.

- **Your Journey So Far** looks back over the road from a single neuron to this project, and forward to the books that come next.